# PhysREVE — Leadfield Physics Visualization

This notebook visualises **why the leadfield attention bias helps** by comparing:

- **REVE** — standard self-attention, no physics knowledge
- **PhysREVE** — same attention + $\alpha B$ where $B = L_{\text{row}}\,L_{\text{row}}^\top$

The forward model of EEG is:
$$\mathbf{y} = L \cdot \mathbf{s} + \varepsilon$$

| Symbol | Shape | Meaning |
|---|---|---|
| $\mathbf{y}$ | $(C \times T)$ | Observed EEG at $C$ electrodes |
| $L$ | $(C \times N)$ | Leadfield — maps $N$ brain sources to electrodes |
| $\mathbf{s}$ | $(N \times T)$ | Latent cortical source activations |

The PhysREVE attention bias is:
$$B_{ij} = (L_{\text{row}})_i \cdot (L_{\text{row}})_j = \cos(\ell_i,\, \ell_j) \in [-1, 1]$$

$B_{ij} \approx 1$ means electrodes $i$ and $j$ see the same cortical sources — the transformer should attend strongly between them.

In [1]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo = '/content/PhysREVE'
    if not os.path.exists(repo):
        subprocess.check_call(['git', 'clone', '-q', 'https://github.com/UgoBruzadin/PhysREVE.git', repo])
    else:
        subprocess.check_call(['git', '-C', repo, 'pull', '-q'])
    if repo not in sys.path:
        sys.path.insert(0, repo)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mne>=1.6', 'moabb>=1.0', 'xgboost'])
    print('Colab environment ready.')
else:
    print('Local environment — ensure you ran: pip install -e .')

Colab environment ready.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from physreve.physics import build_leadfield, motor_roi_indices
from physreve.datasets.bciv2a import CH_NAMES, SFREQ

N_CH = len(CH_NAMES)
print(f'Channels: {N_CH}  ({CH_NAMES[:5]} ...)')

Channels: 22  (['Fz', 'FC3', 'FC1', 'FCz', 'FC2'] ...)


## 1. Build Leadfield

Uses MNE's 3-shell sphere conductor model — no individual MRI required.
The sphere is fitted automatically to the electrode positions.

- **Shell 1 — Brain:** innermost, cortical sources live here
- **Shell 2 — Skull:** high-resistance barrier, smears source signals
- **Shell 3 — Scalp:** where electrodes sit

We also get back the **raw leadfield** $L$ for visualisation.

In [3]:
L_col_np, L_row_np, src_pos, info, L_raw = build_leadfield(
    CH_NAMES, sfreq=SFREQ, verbose=True, return_raw=True
)

# Attention bias matrix  B = L_row @ L_row^T
B = L_row_np @ L_row_np.T   # (n_ch, n_ch) cosine similarity

# Motor ROI source indices
lh_idx_np, rh_idx_np = motor_roi_indices(info, src_pos, CH_NAMES)

# Electrode positions in metres
elec_np = np.array([info['chs'][i]['loc'][:3] for i in range(N_CH)])

print(f'\nLeadfield L        : {L_raw.shape}  (n_ch × n_src)')
print(f'Attention bias B   : {B.shape}  (n_ch × n_ch)')
print(f'Source positions   : {src_pos.shape}')
print(f'LH motor sources   : {len(lh_idx_np)}')
print(f'RH motor sources   : {len(rh_idx_np)}')

  Source space: 1496 active dipoles
  Leadfield shape: (22, 1496)

Leadfield L        : (22, 1496)  (n_ch × n_src)
Attention bias B   : (22, 22)  (n_ch × n_ch)
Source positions   : (1496, 3)
LH motor sources   : 84
RH motor sources   : 83


## 2. Visualization

Three panels telling a single physics story:

1. **3-Shell sphere model** — how a cortical source's signal reaches the scalp electrodes (the physics $\mathbf{y} = L\mathbf{s}$)
2. **Leadfield matrix $L$** — each row is one electrode's sensitivity profile across all brain sources; similar rows mean correlated electrodes
3. **Attention bias $B = L_{\text{row}}L_{\text{row}}^\top$** — dot-product of rows from ②; this is what gets added to the transformer attention logits in PhysREVE

> REVE skips steps ② and ③ entirely — its transformer has no knowledge of this structure.

In [12]:
# ── Helper: draw a transparent sphere ────────────────────────────────────────
def _draw_sphere(ax, r, ctr, color, alpha, n=28):
    u, v = np.mgrid[0:2*np.pi:n*1j, 0:np.pi:n*1j]
    x = r * np.cos(u) * np.sin(v) + ctr[0]
    y = r * np.sin(u) * np.sin(v) + ctr[1]
    z = r * np.cos(v)              + ctr[2]
    ax.plot_surface(x, y, z, color=color, alpha=alpha,
                    linewidth=0, antialiased=True, shade=True)

# ── Sort sources left→right for readable leadfield heatmap ───────────────────
src_order = np.argsort(src_pos[:, 0])   # sort by x (left hemisphere first)

# ── Figure ────────────────────────────────────────────────────────────────────
BG = '#0d1117'
fig = plt.figure(figsize=(22, 8))
fig.patch.set_facecolor(BG)
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.14, left=0.04, right=0.97)

ax1 = fig.add_subplot(gs[0], projection='3d')
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])

for ax in [ax2, ax3]:
    ax.set_facecolor(BG)

# ── Panel 1: 3-Shell sphere + leadfield projections ──────────────────────────
ax1.set_facecolor(BG)
ax1.xaxis.pane.fill = ax1.yaxis.pane.fill = ax1.zaxis.pane.fill = False
for spine in [ax1.xaxis, ax1.yaxis, ax1.zaxis]:
    spine.pane.set_edgecolor('#222')

ax1.set_title('① 3-Shell Head Model\nLeadfield projections (source → electrode)',
               color='white', fontsize=11, pad=8)

elec_np_loc = np.array([info['chs'][i]['loc'][:3] for i in range(N_CH)])
_ctr    = elec_np_loc.mean(axis=0)
_dists  = np.linalg.norm(elec_np_loc - _ctr, axis=1)
r_scalp = _dists.max() * 1.04
r_skull = r_scalp * 0.956
r_brain = r_scalp * 0.878

_draw_sphere(ax1, r_scalp, _ctr, '#f4a460', 0.05)   # scalp
_draw_sphere(ax1, r_skull, _ctr, '#aaaaaa', 0.09)   # skull
_draw_sphere(ax1, r_brain, _ctr, '#4a90d9', 0.15)   # brain

ax1.scatter(src_pos[:,0], src_pos[:,1], src_pos[:,2],
            c='#7eb8f7', s=2, alpha=0.20, linewidths=0)

ax1.scatter(elec_np_loc[:,0], elec_np_loc[:,1], elec_np_loc[:,2],
            c='#ff4444', s=50, zorder=6, linewidths=0, label='Electrodes')

for name in ['C3', 'C4', 'Cz', 'Fz']:
    if name in CH_NAMES:
        i = CH_NAMES.index(name)
        v = elec_np_loc[i] - _ctr
        ax1.text(*(_ctr + v * 1.22).tolist(),
                 name, color='#ffaaaa', fontsize=7)

for roi_idx, clr, lbl in [
    (lh_idx_np, '#ff8c00', 'LH motor (C3)'),
    (rh_idx_np, '#39ff14', 'RH motor (C4)'),
]:
    picks = roi_idx[:3]
    ax1.scatter(src_pos[picks,0], src_pos[picks,1], src_pos[picks,2],
                c=clr, s=90, zorder=7, label=lbl, edgecolors='white', linewidths=0.5)
    for si in picks:
        col     = L_col_np[:, si]
        col_abs = np.abs(col)
        thresh  = col_abs.max() * 0.22
        strong  = np.where(col_abs > thresh)[0]
        if len(strong) == 0:
            continue
        norms = col_abs[strong] / col_abs[strong].max()
        for ei, nv in zip(strong, norms):
            ax1.plot([src_pos[si,0], elec_np_loc[ei,0]],
                     [src_pos[si,1], elec_np_loc[ei,1]],
                     [src_pos[si,2], elec_np_loc[ei,2]],
                     c=clr, alpha=float(nv) * 0.70, lw=1.3)

_lim = r_scalp * 1.30
ax1.set_xlim(_ctr[0]-_lim, _ctr[0]+_lim)
ax1.set_ylim(_ctr[1]-_lim, _ctr[1]+_lim)
ax1.set_zlim(_ctr[2]-_lim, _ctr[2]+_lim)
ax1.tick_params(colors='#666', labelsize=6)
ax1.set_xlabel('x (m)', color='#888', fontsize=8, labelpad=2)
ax1.set_ylabel('y (m)', color='#888', fontsize=8, labelpad=2)
ax1.set_zlabel('z (m)', color='#888', fontsize=8, labelpad=2)
ax1.legend(fontsize=7.5, loc='upper left',
           facecolor='#1a1a2e', labelcolor='white', framealpha=0.75)

for txt, clr, y in [('Scalp','#f4a460',0.77),('Skull','#aaaaaa',0.71),('Brain','#7eb8f7',0.65)]:
    ax1.text2D(0.70, y, txt, transform=ax1.transAxes, color=clr, fontsize=9, fontweight='bold')

# ── Panel 2: Raw leadfield matrix L_row (n_ch × n_src) ───────────────────────
# Show all sources sorted left→right; each row is one electrode's sensitivity profile
im2 = ax2.imshow(L_row_np[:, src_order], aspect='auto', cmap='RdBu_r',
                  vmin=-1, vmax=1, interpolation='nearest')
ax2.set_title('② Leadfield Matrix  $L$\n(row-norm. — electrode sensitivity to every source)',
               color='white', fontsize=11, pad=8)
ax2.set_xlabel('Brain source  (sorted left → right hemisphere)', color='#aaa', fontsize=9)
ax2.set_ylabel('EEG electrode', color='#aaa', fontsize=9)
ax2.set_yticks(range(N_CH))
ax2.set_yticklabels(CH_NAMES, fontsize=6, color='#ccc')
ax2.set_xticks([0, len(src_order)//2, len(src_order)-1])
ax2.set_xticklabels(['Left\nhemisphere', 'Midline', 'Right\nhemisphere'],
                     fontsize=8, color='#ccc')
ax2.tick_params(colors='#444')
for sp in ax2.spines.values(): sp.set_edgecolor('#333')

cb2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, label='Sensitivity')
cb2.ax.yaxis.set_tick_params(color='#aaa')
cb2.ax.yaxis.label.set_color('#aaa')
plt.setp(cb2.ax.yaxis.get_ticklabels(), color='#aaa')

ax2.text(0.5, -0.12,
    "Each row = one electrode's fingerprint across all brain sources\n"
    "C3 and CP3 rows look similar — they see the same cortex",
    ha='center', transform=ax2.transAxes, fontsize=8.5, color='#8899aa', style='italic')

# ── Panel 3: Attention bias B = L_row @ L_row^T ──────────────────────────────
im3 = ax3.imshow(B, aspect='equal', cmap='RdBu_r', vmin=-1, vmax=1,
                  interpolation='nearest')
ax3.set_title(
    '③ PhysREVE Attention Bias  $B = L_{\\mathrm{row}}\\,L_{\\mathrm{row}}^\\top$\n'
    'Electrode-electrode cosine similarity (what the transformer uses)',
    color='white', fontsize=11, pad=8)
ax3.set_xlabel('Electrode j', color='#aaa', fontsize=10)
ax3.set_ylabel('Electrode i', color='#aaa', fontsize=10)
ax3.set_xticks(range(N_CH))
ax3.set_xticklabels(CH_NAMES, fontsize=5.5, rotation=90, color='#ccc')
ax3.set_yticks(range(N_CH))
ax3.set_yticklabels(CH_NAMES, fontsize=5.5, color='#ccc')
ax3.tick_params(colors='#444')
for sp in ax3.spines.values(): sp.set_edgecolor('#333')

for (ni, nj) in [('C3','C4'), ('C3','C1'), ('C4','C2'), ('Cz','CPz'), ('Fz','FCz')]:
    if ni in CH_NAMES and nj in CH_NAMES:
        ii, jj = CH_NAMES.index(ni), CH_NAMES.index(nj)
        ax3.add_patch(plt.Rectangle((jj-0.5, ii-0.5), 1, 1,
                                     fill=False, edgecolor='#ffd700', lw=1.8, zorder=5))

cb3 = plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04, label='Cosine similarity')
cb3.ax.yaxis.set_tick_params(color='#aaa')
cb3.ax.yaxis.label.set_color('#aaa')
plt.setp(cb3.ax.yaxis.get_ticklabels(), color='#aaa')

ax3.text(0.5, -0.12,
    r'$B_{ij}$ = dot product of rows $i$ and $j$ from ②' + '\n'
    r'Red = share brain sources → attend more  |  Blue = independent sources',
    ha='center', transform=ax3.transAxes, fontsize=8.5, color='#8899aa', style='italic')

# ── Shared title ──────────────────────────────────────────────────────────────
fig.suptitle(
    'PhysREVE Leadfield Physics  —  How $\\mathbf{y} = L \\cdot \\mathbf{s} + \\varepsilon$ becomes an attention prior\n'
    'REVE ignores this structure entirely; PhysREVE bakes it into every attention layer',
    color='white', fontsize=12, fontweight='bold', y=1.02
)

plt.savefig('physreve_leadfield_vis.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved physreve_leadfield_vis.png')

## 3. How to Read the Plots

### Panel ①: 3-Shell Sphere Model
- **Red dots** = EEG electrodes on the scalp
- **Blue cloud** = all brain source dipoles
- **Orange / green lines** = how LH and RH motor sources project to electrodes — each line's opacity is proportional to that electrode's sensitivity to the source
- The skull shell acts as a resistive barrier, spreading each source's signal across many electrodes

### Panel ②: Leadfield Matrix $L$
- Rows = electrodes, columns = brain sources (sorted left→right hemisphere)
- **Red** = electrode is positively sensitive to that source; **Blue** = negatively sensitive
- Notice that nearby electrodes (e.g. C3, C1, CP3) have very similar row patterns — they see the same cortex. This is the structure PhysREVE exploits.
- REVE has no access to this matrix — it treats every electrode as independent

### Panel ③: Attention Bias $B = L_{\text{row}}L_{\text{row}}^\top$
- Each cell $(i,j)$ = cosine similarity between electrode $i$'s and $j$'s leadfield rows from ②
- **Red (warm)** = high similarity → these electrodes share brain sources → the transformer is biased to attend between them
- **Blue (cool)** = low/negative similarity → independent source populations
- **Golden boxes** mark anatomically meaningful pairs:
  - C3↔C4 (contralateral motor cortex — key for left/right hand MI)
  - C3↔C1, C4↔C2 (ipsilateral motor, same hemisphere)
  - Cz↔CPz (central-parietal, motor + somatosensory)
  - Fz↔FCz (frontal motor planning)

### The core argument
Panel ② → Panel ③ is a matrix multiplication. PhysREVE adds $\alpha B$ to the raw attention logits before softmax, so electrodes that share cortical sources automatically have a higher attention score — before any data is seen. REVE must learn this geometry from scratch during pretraining, which requires far more data and epochs.